# Avaliação da recuperação sobre a Gold — Databricks

Este notebook lê `cognus.gold.vw_rag_conhecimento` com Spark e registra os experimentos
no MLflow do Workspace. Não depende do Ollama para a busca BM25.

O `localhost` do cluster não é o Mac do usuário. A geração pelo Qwen instalado no
Mac está no notebook local 02. Não exponha o Ollama à internet para executar este notebook.
As referências iniciais são rascunhos: sem anotação humana, não haverá Precision/Recall reais.

Execute no Git folder deste repositório ou na cópia do projeto em `/Workspace/Users/<usuario>/rag_cognus`.

In [ ]:
%pip install "langchain-core>=1,<2" "langchain-ollama>=1,<2" "mlflow>=3.6,<4" "rank-bm25>=0.2.2,<0.3" "databricks-sdk>=0.70,<1" "python-dotenv>=1,<2"

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pathlib import Path
import sys
from dataclasses import replace

# Git folder: procura o projeto a partir da pasta atual.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/rag_cognus").is_dir()), None)
if PROJECT_ROOT is None:
    user = spark.sql("SELECT current_user() AS user").first().user
    PROJECT_ROOT = Path("/Workspace/Users") / user / "rag_cognus"
if not (PROJECT_ROOT / "src/rag_cognus").is_dir():
    raise FileNotFoundError("Ajuste PROJECT_ROOT para a pasta deste repositório no Workspace.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from rag_cognus.config import Settings
from rag_cognus.gold import load_from_spark
from rag_cognus.retrieval import GoldRetriever
from rag_cognus.evaluation import evaluate, load_cases

user = spark.sql("SELECT current_user() AS user").first().user
settings = replace(Settings(), tracking_uri="databricks",
                   experiment=f"/Users/{user}/rag_cognus_avaliacoes")
documents = load_from_spark(spark, settings.gold_view)
print("Trechos elegíveis lidos da Gold:", len(documents))
cases = load_cases(PROJECT_ROOT / "data/evaluation/questions.jsonl")

In [ ]:
summaries = []
for k in (3, 5):
    config = replace(settings, backend="bm25", top_k=k)
    retriever = GoldRetriever(documents, config)
    summary = evaluate(cases, retriever, config,
                       output_dir="/tmp/rag-cognus-evaluation",
                       run_name=f"gold-bm25-k{k}")
    summaries.append(summary)
    print(summary)
if any(s["metrics"]["error_rate"] for s in summaries):
    raise RuntimeError("Há erros por pergunta; consulte per_case.json no MLflow.")

## Como interpretar

Abra o experimento `/Users/<usuario>/rag_cognus_avaliacoes` e compare os runs.
`niche_metadata_precision_proxy` mede concordância com rótulos da própria Gold,
não relevância semântica. Como as consultas não recebem o nicho esperado como
filtro, não usamos a resposta esperada para facilitar a recuperação.

`precision_at_k`, `recall_at_k`, `hit_at_k` e `mrr_at_k` só aparecem quando há
`label_status=human_verified` e referências de trechos que existem neste snapshot.
Revise candidatos e edite o dataset antes de comparar qualidade real.

O snapshot pode ser salvo com `save_snapshot` para levar os mesmos trechos à
execução local. O comando local `rag-cognus sync-gold` já faz essa leitura diretamente
via SQL Warehouse, sem publicar os documentos no GitHub.